In [30]:
import torch
from torch.utils.data import Dataset
import os

class BeatmapChunkDataset(Dataset):
    def __init__(self, input_folder):
        self.audio_folder = os.path.join(input_folder, "audio")
        
        df = pd.read_csv(os.path.join(input_folder, "chunked.csv"))
        self.groups = list(df.groupby(["id", "chunk_id"]))

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        (beatmap_id, chunk_id), group = self.groups[idx]

        features = torch.tensor(
            group[["type_circle", "type_slider", "type_spinner", 
                   "hit_start_rel", "hit_end_rel"]].values, dtype=torch.float32)
        beatmapset_id = beatmap_id.split("-")[0]
        chunk_audio_path = os.path.join(self.audio_folder, f"{beatmapset_id}_chunk{chunk_id}.pt")
        
        difficulty_rating = torch.tensor([group.iloc[0]["difficulty_rating"]], dtype=torch.float32)
        
        return {
            "beatmap_id": beatmap_id,
            "chunk_id": chunk_id,
            "features": features,
            "audio": torch.load(chunk_audio_path),
            "difficulty_rating": difficulty_rating 
        }


In [61]:
import torch
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    beatmap_ids = [item["beatmap_id"] for item in batch]
    chunk_ids = [item["chunk_id"] for item in batch]

    features_list = [item["features"] for item in batch]
    features_padded = pad_sequence(features_list, batch_first=True, padding_value=0.0)

    features_mask = torch.zeros(features_padded.shape[:2], dtype=torch.bool)
    for i, feat in enumerate(features_list):
        features_mask[i, :feat.shape[0]] = 1

    audio_embeddings = torch.stack([item["audio"].squeeze(0) for item in batch], dim=0)
    difficulty_ratings = torch.tensor([item["difficulty_rating"] for item in batch], dtype=torch.float).unsqueeze(1)

    return {
        "beatmap_ids": beatmap_ids,
        "chunk_ids": torch.tensor(chunk_ids, dtype=torch.long),
        "features": features_padded,
        "features_mask": features_mask,
        "audio": audio_embeddings,
        "difficulty_rating" : difficulty_ratings
    }


In [62]:
import pandas as pd
from torch.utils.data import DataLoader

input_folder = "/home/saliherdemk/try_dataset/chunked"
dataset = BeatmapChunkDataset(input_folder)

dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn = collate_fn)

for batch in dataloader:
    print(batch['audio'][0].shape)
    print(batch['features'].shape)
    break

torch.Size([749, 768])
torch.Size([2, 63, 5])


# Model

In [94]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, input_dim=768, emb_dim=1023, max_seq_len=1000, nhead=8, num_layers=6, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=256, kernel_size=1)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(in_channels=256, out_channels=emb_dim, kernel_size=1)

        d_model = emb_dim + 1

        self.pos_embedding = nn.Parameter(torch.randn(1, max_seq_len, d_model))  # [1, seq_len, d_model]
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x, diff_rating):
        # x: [batch, seq_len, d_model]
        x = x.permute(0, 2, 1)  # [batch, d_model, seq_len]
        x = self.conv1(x)
        x = self.relu(x)
        x = self.conv2(x) # [batch, d_model, seq_len]
        x = x.permute(0, 2, 1) # [batch, seq_len, d_model]
    
        seq_len = x.size(1)
        diff_rating = diff_rating.unsqueeze(1).expand(-1, seq_len, -1)
        x = torch.cat([x, diff_rating], dim=-1)

        x = x + self.pos_embedding[:, :seq_len, :]
        out = self.encoder(x)
    
        return x



In [95]:
encoder = Encoder()
tEncoder = TransformerEncoder()
for batch in dataloader:
    audio_batch = batch["audio"]
    diff_batch = batch["difficulty_rating"]
    e = encoder(audio_batch, diff_batch)
    print(e.shape)
    break

torch.Size([2, 749, 1024])
